# 🛡️ Federated Learning + AE-WGAN — NSL-KDD

### Configuration :
- **Features** : Table 5 exacte (24 features incl. difficulty_level)
- **Architecture** : GitHub originale (BiGAN)
- **Oversampling** : 10% de Normal
- **FL** : 4 clients non-IID, FedAvg, 5 rounds
- **Comparaison** : Baseline vs Federated AE-WGAN

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q tensorflow numpy pandas scikit-learn matplotlib scipy

In [ ]:
import os, urllib.request, time, copy
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Reshape, Flatten, Concatenate)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
SEED = 42; np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"TF: {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")

## 1. Preprocessing (Table 5)

In [ ]:
url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
for f,u in {"KDDTrain+.txt":url+"KDDTrain%2B.txt","KDDTest+.txt":url+"KDDTest%2B.txt"}.items():
    if not os.path.exists(f): urllib.request.urlretrieve(u, f)

COLS = ['duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files','num_outbound_cmds',
    'is_host_login','is_guest_login','count','srv_count','serror_rate',
    'srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate',
    'diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty_level']

ATK_MAP = {'normal':'Normal',
    'back':'DoS','land':'DoS','neptune':'DoS','pod':'DoS','smurf':'DoS','teardrop':'DoS',
    'mailbomb':'DoS','apache2':'DoS','processtable':'DoS','udpstorm':'DoS',
    'satan':'Probe','ipsweep':'Probe','nmap':'Probe','portsweep':'Probe','mscan':'Probe','saint':'Probe',
    'guess_passwd':'R2L','ftp_write':'R2L','imap':'R2L','phf':'R2L','multihop':'R2L',
    'warezmaster':'R2L','warezclient':'R2L','spy':'R2L','xlock':'R2L','xsnoop':'R2L',
    'snmpguess':'R2L','snmpgetattack':'R2L','httptunnel':'R2L','sendmail':'R2L','named':'R2L','worm':'R2L',
    'buffer_overflow':'U2R','loadmodule':'U2R','rootkit':'U2R','perl':'U2R',
    'sqlattack':'U2R','xterm':'U2R','ps':'U2R'}

PAPER_FEATURES = ['protocol_type','flag','land','wrong_fragment','logged_in',
    'count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate',
    'same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_srv_rerror_rate','difficulty_level']

df_tr = pd.read_csv("KDDTrain+.txt",header=None,names=COLS)
df_te = pd.read_csv("KDDTest+.txt",header=None,names=COLS)
df_tr['label']=df_tr['label'].str.strip().str.rstrip('.')
df_te['label']=df_te['label'].str.strip().str.rstrip('.')

y_train_cat=df_tr['label'].map(ATK_MAP).fillna('Unknown')
y_test_cat=df_te['label'].map(ATK_MAP).fillna('Unknown')
y_train_detail=df_tr['label'].copy()

X_tr_df=df_tr[PAPER_FEATURES].copy(); X_te_df=df_te[PAPER_FEATURES].copy()
cat_cols=X_tr_df.select_dtypes(include=['object']).columns.tolist()
enc=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
X_tr_df[cat_cols]=enc.fit_transform(X_tr_df[cat_cols])
X_te_df[cat_cols]=enc.transform(X_te_df[cat_cols])
scaler=MinMaxScaler()
X_train_final=scaler.fit_transform(X_tr_df.values)
X_test_final=scaler.transform(X_te_df.values)
INPUT_DIM=24
N_NORMAL=(y_train_cat=='Normal').sum()
TARGET=int(N_NORMAL*0.10)
print(f"✅ Train:{X_train_final.shape} Test:{X_test_final.shape} Seuil:{TARGET}")

## 2. AE-WGAN (architecture GitHub + FL support)

In [ ]:
class AEWGAN_FL:
    """AE-WGAN avec get_weights/set_weights pour Federated Learning."""
    def __init__(self):
        self.img_rows=4; self.img_cols=6; self.channels=1
        self.img_shape=(4,6,1); self.latent_dim=32
        self.flat_dim=int(np.prod(self.img_shape))
        opt=tf.keras.optimizers.Adamax(0.00009)
        self.discriminator=self._build_disc()
        self.discriminator.compile(loss=self._wloss,optimizer=opt,metrics=['accuracy'])
        self.generator=self._build_gen()
        self.encoder=self._build_enc()
        self.discriminator.trainable=False
        z=Input(shape=(self.latent_dim,)); img_=self.generator(z)
        img=Input(shape=self.img_shape); z_=self.encoder(img)
        fake=self.discriminator([z,img_]); valid=self.discriminator([z_,img])
        self.combined=Model([z,img],[fake,valid])
        self.combined.compile(loss=[self._wloss,self._wloss],optimizer=opt)

    def _wloss(self,y_true,y_pred): return K.mean(y_true*y_pred)
    def _build_enc(self):
        m=Sequential([Flatten(input_shape=self.img_shape),Dense(20,activation='relu'),Dense(self.latent_dim)])
        img=Input(shape=self.img_shape); return Model(img,m(img),name='encoder')
    def _build_gen(self):
        m=Sequential([Dense(20,input_dim=self.latent_dim,activation='relu'),Dense(40,activation='relu'),
            Dense(80,activation='relu'),Dense(120,activation='relu'),Dense(160,activation='relu'),
            Dense(self.flat_dim,activation='relu'),Reshape(self.img_shape)])
        z=Input(shape=(self.latent_dim,)); return Model(z,m(z),name='generator')
    def _build_disc(self):
        z=Input(shape=(self.latent_dim,)); img=Input(shape=self.img_shape)
        d_in=Concatenate()([z,Flatten()(img)])
        x=Dense(80,activation='relu')(d_in); x=Dense(40,activation='relu')(x)
        v=Dense(1,activation='sigmoid')(x); return Model([z,img],v,name='discriminator')

    def train(self,X,epochs=20,batch_size=32,verbose=0):
        reshaped=X.reshape(-1,4,6,1); n=reshaped.shape[0]
        if n<batch_size: batch_size=max(n//2,2)
        valid=np.ones((batch_size,1)); fake=np.zeros((batch_size,1))
        for epoch in range(epochs):
            z=np.random.normal(0,1,size=(batch_size,self.latent_dim))
            imgs_=self.generator.predict(z,verbose=0)
            idx=np.random.randint(0,n,batch_size); imgs=reshaped[idx]
            z_=self.encoder.predict(imgs,verbose=0)
            self.discriminator.train_on_batch([z_,imgs],valid)
            self.discriminator.train_on_batch([z,imgs_],fake)
            self.combined.train_on_batch([z,imgs],[valid,fake])

    def generate_batch(self,n):
        noise=np.random.normal(0,1,size=(n,self.latent_dim))
        return self.generator.predict(noise,verbose=0,batch_size=256).reshape(-1,self.flat_dim)

    def get_fl_weights(self):
        return {'gen':self.generator.get_weights(),
                'enc':self.encoder.get_weights(),
                'disc':self.discriminator.get_weights()}

    def set_fl_weights(self,w):
        self.generator.set_weights(w['gen'])
        self.encoder.set_weights(w['enc'])
        self.discriminator.set_weights(w['disc'])

    def init_weights(self):
        """Forward pass pour initialiser les poids."""
        dummy_img=np.zeros((2,4,6,1),dtype=np.float32)
        dummy_z=np.zeros((2,32),dtype=np.float32)
        self.generator.predict(dummy_z,verbose=0)
        self.encoder.predict(dummy_img,verbose=0)
        self.discriminator.predict([dummy_z,dummy_img],verbose=0)

print("✅ AEWGAN_FL défini")

## 3. FedAvg

In [ ]:
def fedavg_aggregate(client_weights_list, client_sizes):
    """Moyenne pondérée des poids (FedAvg)."""
    total = sum(client_sizes)
    fracs = [s/total for s in client_sizes]
    global_w = {}
    for component in ['gen','enc','disc']:
        n_layers = len(client_weights_list[0][component])
        agg = []
        for li in range(n_layers):
            ws = np.zeros_like(client_weights_list[0][component][li])
            for ci, cw in enumerate(client_weights_list):
                ws += fracs[ci] * cw[component][li]
            agg.append(ws)
        global_w[component] = agg
    return global_w

print("✅ FedAvg défini")

## 4. Partitionnement Non-IID (4 clients)

In [ ]:
def partition_non_iid(X, y_cat, y_detail):
    categories = ['DoS','Probe','R2L','U2R']
    normal_idx = np.where(y_cat.values=='Normal')[0]
    np.random.shuffle(normal_idx)
    normal_splits = np.array_split(normal_idx, 4)
    clients = []
    for i, cat in enumerate(categories):
        cat_idx = np.where(y_cat.values==cat)[0]
        client_idx = np.concatenate([cat_idx, normal_splits[i]])
        np.random.shuffle(client_idx)
        clients.append({
            'X': X[client_idx],
            'y_cat': y_cat.iloc[client_idx].reset_index(drop=True),
            'y_detail': y_detail.iloc[client_idx].reset_index(drop=True),
            'name': f'Client {i+1} ({cat})',
            'size': len(client_idx)})
    return clients

clients = partition_non_iid(X_train_final, y_train_cat, y_train_detail)
print("=" * 60)
print("  PARTITIONNEMENT NON-IID")
print("=" * 60)
for c in clients:
    print(f"  {c['name']}: {c['size']} samples")
    print(f"    {dict(c['y_cat'].value_counts())}")

## 5. Entraînement Federated Learning

In [ ]:
N_ROUNDS = 5
LOCAL_EPOCHS = 20  # 20 × 5 rounds = 100 epochs total ≈ 101
MIN_SAMPLES = 10

print("=" * 60)
print(f"  FEDERATED LEARNING — {N_ROUNDS} rounds, {LOCAL_EPOCHS} epochs locales")
print("=" * 60)

fed_synthetic_X = []
fed_synthetic_y = []
start_time = time.time()

# Collecter tous les types d'attaques uniques
all_attacks = set()
for c in clients:
    attacks = c['y_detail'][c['y_cat']!='Normal'].unique()
    all_attacks.update(attacks)

for attack in sorted(all_attacks):
    # Trouver quels clients ont cet attack type avec assez de samples
    participating = []
    total_samples = 0
    for c in clients:
        mask = c['y_detail'] == attack
        n = mask.sum()
        if n >= MIN_SAMPLES:
            participating.append({'name':c['name'], 'X':c['X'][mask.values], 'size':n})
            total_samples += n

    if not participating:
        continue

    # Calcul du nombre à générer (10% de Normal)
    if total_samples >= TARGET:
        print(f"  [{attack}] ⏩ {total_samples} >= {TARGET} — skip")
        continue

    n_to_gen = TARGET - total_samples

    if len(participating) == 1:
        # Un seul client → entraînement local simple
        pc = participating[0]
        print(f"\n  [{attack}] 🧠 Local seul ({pc['name']}, {pc['size']} samples, +{n_to_gen})")
        model = AEWGAN_FL()
        model.train(pc['X'], epochs=LOCAL_EPOCHS*N_ROUNDS, batch_size=min(32,max(pc['size']//2,2)))
        synthetic = model.generate_batch(n_to_gen)
        fed_synthetic_X.append(synthetic)
        fed_synthetic_y.extend([attack]*n_to_gen)
        print(f"  [{attack}] ✅ Généré {n_to_gen} (local)")
        continue

    # Plusieurs clients → FedAvg
    print(f"\n  [{attack}] 🌐 FedAvg ({len(participating)} clients, total {total_samples}, +{n_to_gen})")
    for pc in participating:
        print(f"    - {pc['name']}: {pc['size']}")

    # Initialiser modèle global
    global_model = AEWGAN_FL()
    global_model.init_weights()
    global_weights = global_model.get_fl_weights()

    for rnd in range(N_ROUNDS):
        client_weights_list = []
        client_sizes = []
        for pc in participating:
            local_model = AEWGAN_FL()
            local_model.init_weights()
            local_model.set_fl_weights(global_weights)
            bs = min(32, max(pc['size']//2, 2))
            local_model.train(pc['X'], epochs=LOCAL_EPOCHS, batch_size=bs)
            client_weights_list.append(local_model.get_fl_weights())
            client_sizes.append(pc['size'])
        global_weights = fedavg_aggregate(client_weights_list, client_sizes)
        print(f"    Round {rnd+1}/{N_ROUNDS} ✓")

    # Générer avec le modèle global
    global_model.set_fl_weights(global_weights)
    synthetic = global_model.generate_batch(n_to_gen)
    fed_synthetic_X.append(synthetic)
    fed_synthetic_y.extend([attack]*n_to_gen)
    print(f"  [{attack}] ✅ Généré {n_to_gen} (federated)")

elapsed = (time.time()-start_time)/60
print(f"\n--- {elapsed:.1f} minutes ---")

# Combiner
if fed_synthetic_X:
    X_fed_syn = np.vstack(fed_synthetic_X)
    y_fed_syn = pd.Series(fed_synthetic_y)
    X_fed_aug = np.vstack([X_train_final, X_fed_syn])
    y_fed_detail = pd.concat([y_train_detail, y_fed_syn], ignore_index=True)
    y_fed_cat = y_fed_detail.map(ATK_MAP).fillna('Unknown')
else:
    X_fed_aug = X_train_final; y_fed_cat = y_train_cat

print(f"\n  Original:    {X_train_final.shape[0]:>10}")
print(f"  FL Synth:    {X_fed_aug.shape[0]-X_train_final.shape[0]:>10}")
print(f"  Total:       {X_fed_aug.shape[0]:>10}")

## 6. Classifieurs + Évaluation

In [ ]:
def build_dt(nc=2): return DecisionTreeClassifier(max_depth=7 if nc==2 else 9,random_state=42)
def build_rf(nc=2): return RandomForestClassifier(100,max_depth=8 if nc==2 else 10,random_state=42,n_jobs=-1)
def build_cnn(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential([layers.Input(shape=(dim,1)),layers.Conv1D(40,3,activation='relu',padding='same'),
        layers.Dropout(0.1),layers.Flatten(),layers.Dense(40,activation='relu'),
        layers.Dense(out,activation=act)],name='CNN')
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m
def build_gru(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='GRU'); m.add(layers.Input(shape=(dim,1)))
    if nc==2:
        m.add(layers.GRU(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(10,activation='relu'));m.add(layers.Dense(10,activation='relu'))
    else:
        m.add(layers.GRU(30,activation='relu',return_sequences=True));m.add(layers.Dropout(0.1))
        m.add(layers.GRU(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(30,activation='relu'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

def prepare_labels(y_tr,y_te,binary=True):
    le=LabelEncoder()
    if binary: return (y_tr!='Normal').astype(int).values,(y_te!='Normal').astype(int).values,le
    else:
        le.fit(list(set(y_tr.unique())|set(y_te.unique())))
        return le.transform(y_tr),le.transform(y_te),le

def train_and_eval(name,clf,X_tr,y_tr,X_te,y_te):
    if name in ['CNN','GRU']:
        X_tr3=X_tr.reshape(-1,X_tr.shape[1],1); X_te3=X_te.reshape(-1,X_te.shape[1],1)
        es=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
        clf.fit(X_tr3,y_tr,epochs=50,batch_size=256,validation_split=0.1,callbacks=[es],verbose=0)
        yp=clf.predict(X_te3,verbose=0)
        y_pred=(yp.flatten()>0.5).astype(int) if yp.shape[-1]==1 else np.argmax(yp,axis=1)
    else: clf.fit(X_tr,y_tr); y_pred=clf.predict(X_te)
    avg='binary' if len(np.unique(y_te))==2 else 'weighted'
    return {'classifier':name,'accuracy':accuracy_score(y_te,y_pred),
            'precision':precision_score(y_te,y_pred,average=avg,zero_division=0),
            'recall':recall_score(y_te,y_pred,average=avg,zero_division=0),
            'f1_score':f1_score(y_te,y_pred,average=avg,zero_division=0)}

def run_clf(X_tr,y_cat,X_te,y_te_cat,dim,binary=True,label=''):
    y_tr,y_te,le=prepare_labels(y_cat,y_te_cat,binary)
    nc=2 if binary else len(np.unique(y_tr))
    builders={'DT':lambda:build_dt(nc),'RF':lambda:build_rf(nc),
              'CNN':lambda:build_cnn(dim,nc),'GRU':lambda:build_gru(dim,nc)}
    results=[]
    for name,builder in builders.items():
        print(f"    {label} {name}...",end=" ")
        r=train_and_eval(name,builder(),X_tr,y_tr,X_te,y_te)
        print(f"Acc={r['accuracy']:.4f} F1={r['f1_score']:.4f}")
        results.append(r)
    return results

print("✅ Classifieurs prêts (DT, RF, CNN, GRU)")

In [ ]:
print("\n" + "="*60)
print("  CLASSIFICATION BINAIRE")
print("="*60)
print("\n  --- Baseline ---")
base_bin=run_clf(X_train_final,y_train_cat,X_test_final,y_test_cat,INPUT_DIM,binary=True,label='[Base]')
print("\n  --- Federated AE-WGAN ---")
fed_bin=run_clf(X_fed_aug,y_fed_cat,X_test_final,y_test_cat,INPUT_DIM,binary=True,label='[FL]')

print("\n" + "="*60)
print("  CLASSIFICATION MULTICLASSE")
print("="*60)
print("\n  --- Baseline ---")
base_multi=run_clf(X_train_final,y_train_cat,X_test_final,y_test_cat,INPUT_DIM,binary=False,label='[Base]')
print("\n  --- Federated AE-WGAN ---")
fed_multi=run_clf(X_fed_aug,y_fed_cat,X_test_final,y_test_cat,INPUT_DIM,binary=False,label='[FL]')

## 7. Résultats et Export

In [ ]:
print("\n" + "="*70)
print("  COMPARAISON: BASELINE vs FEDERATED AE-WGAN")
print("="*70)
for task,br,fr in [("BINAIRE",base_bin,fed_bin),("MULTICLASSE",base_multi,fed_multi)]:
    print(f"\n  --- {task} ---")
    print(f"  {'Clf':<6} {'Base Acc':>9} {'FL Acc':>9} {'Δ Acc':>7}  {'Base F1':>9} {'FL F1':>9} {'Δ F1':>7}")
    print(f"  {'─'*58}")
    for b,f in zip(br,fr):
        da=f['accuracy']-b['accuracy']; df=f['f1_score']-b['f1_score']
        print(f"  {b['classifier']:<6} {b['accuracy']:>9.4f} {f['accuracy']:>9.4f} {'+'if da>=0 else ''}{da:>6.4f}  "
              f"{b['f1_score']:>9.4f} {f['f1_score']:>9.4f} {'+'if df>=0 else ''}{df:>6.4f}")

# Visualization
fig,axes=plt.subplots(1,2,figsize=(14,6))
fig.suptitle('NSL-KDD: Baseline vs Federated AE-WGAN',fontsize=14,fontweight='bold')
for idx,(task,br,fr) in enumerate([('Binary',base_bin,fed_bin),('Multiclass',base_multi,fed_multi)]):
    ax=axes[idx]
    clfs=[r['classifier'] for r in br]; x=np.arange(len(clfs)); w=0.35
    ax.bar(x-w/2,[r['f1_score'] for r in br],w,label='Baseline',color='#3498db',alpha=0.8)
    ax.bar(x+w/2,[r['f1_score'] for r in fr],w,label='FL AE-WGAN',color='#e74c3c',alpha=0.8)
    ax.set_xlabel('Classifieur');ax.set_ylabel('F1 Score');ax.set_title(f'{task}')
    ax.set_xticks(x);ax.set_xticklabels(clfs);ax.legend();ax.set_ylim(0.5,1.05);ax.grid(axis='y',alpha=0.3)
    for i,(b,f) in enumerate(zip(br,fr)):
        d=f['f1_score']-b['f1_score']; color='green' if d>=0 else 'red'
        ax.annotate(f'{d:+.2%}',xy=(i+w/2,f['f1_score']+0.01),ha='center',fontsize=9,color=color,fontweight='bold')
plt.tight_layout()
plt.savefig('results_federated.png',dpi=150,bbox_inches='tight')
plt.show()

# CSV + Drive
all_r=[]
for r in base_bin: all_r.append({**r,'type':'Baseline','classification':'Binary'})
for r in fed_bin: all_r.append({**r,'type':'FL-AEWGAN','classification':'Binary'})
for r in base_multi: all_r.append({**r,'type':'Baseline','classification':'Multiclass'})
for r in fed_multi: all_r.append({**r,'type':'FL-AEWGAN','classification':'Multiclass'})
df_r=pd.DataFrame(all_r)
df_r.to_csv('results_federated.csv',index=False)
import shutil
shutil.copy('results_federated.csv','/content/drive/MyDrive/')
shutil.copy('results_federated.png','/content/drive/MyDrive/')
print("\n✅ Résultats sauvegardés et copiés sur Drive")
print(df_r.to_string(index=False))